In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
print("All imported successfully!")

In [ ]:
df=pd.read_csv('cleaned_goodreads_books.csv')
print("Dataset shape: {df.shape}")
df.head()

## Univariate Analysis:

In [ ]:
#Separating numerical and categorical columns
num_cols=df.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols=df.select_dtypes(include=['object','category']).columns.tolist()

print(f"Numerical columns: ({len(num_cols)}):{num_cols}")
print(f"Categorical columns: ({len(cat_cols)}):{cat_cols}")

In [ ]:
df[num_cols].describe().T

In [ ]:
stats_df = pd.DataFrame({
    'mean': df[num_cols].mean(),
    'median': df[num_cols].median(),
    'mode': df[num_cols].mode().iloc[0],
    'std': df[num_cols].std(),
    'variance': df[num_cols].var(),
    'skewness': df[num_cols].skew(),
    'kurtosis': df[num_cols].kurtosis()
})

stats_df

In [ ]:
# Histograms for all numerical variables
df[num_cols].hist(bins=30, figsize=(15, 10), edgecolor='black')
plt.suptitle('Histograms of Numerical Variables', fontsize=16, y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution plots for each numerical variable
for col in num_cols:
    plt.figure(figsize=(10, 4))
    
    plt.subplot(1, 2, 1)
    sns.histplot(df[col], kde=True, bins=30)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    
    plt.subplot(1, 2, 2)
    sns.boxplot(x=df[col])
    plt.title(f'Box Plot of {col}')
    
    plt.tight_layout()
    plt.show()

In [ ]:
#Value counts for each categorical variable
for col in cat_cols:
    print(f"\n{'='*50}")
    print(f"Value counts for: {col}")
    print(f"{'='*50}")
    print(df[col].value_counts())
    print(f"\nPercentage distribution:")
    print(df[col].value_counts(normalize=True) * 100)

In [ ]:
TOP_N = 10
for col in cat_cols[:5]:
    plt.figure(figsize=(14, 5))
    value_counts = df[col].value_counts(dropna=False).head(TOP_N)
    total = value_counts.sum()

    #Bar Plot
    plt.subplot(1, 2, 1)
    ax = sns.barplot(
        x=value_counts.index,
        y=value_counts.values,
        palette="viridis"
    )
    
    plt.xticks(rotation=45)
    plt.title(f'Top {TOP_N} Categories in {col}')
    plt.ylabel("Count")
    plt.xlabel(col)

    for i, v in enumerate(value_counts.values):
        ax.text(i, v, f"{(v/total)*100:.1f}%", 
                ha='center', va='bottom', fontsize=9)

    #Pie Chart
    plt.subplot(1, 2, 2)
    if df[col].nunique() <= 10:
        value_counts.plot(
            kind='pie',
            autopct='%1.1f%%',
            startangle=90,
            cmap="viridis"
        )
        plt.ylabel('')
        plt.title(f'Proportion of {col}')
    else:
        plt.text(0.5, 0.5, 
                 f'Too many categories\n(Showing Top {TOP_N} in bar)',
                 ha='center', va='center')
        plt.axis('off')

    plt.tight_layout()
    plt.show()


# Linear Regression

1. To find relationship between book length and rating

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

x=df[["num_pages"]]
y=df["average_rating"]
model=LinearRegression()
model.fit(x,y)
predictions=model.predict(x)
print("Slope:",model.coef_[0])
print("Intercept:",model.intercept_)
print("R^2 score:",model.score(x,y))

- Book length has almost no effect on rating
- 97.74& of rating variation is caused by other factors

In [ ]:
plt.scatter(x,y,alpha=0.3)
plt.plot(x,predictions, color='red')
plt.xlabel("Number of pages")
plt.ylabel("Average rating")
plt.title("Linear Regression: Pages vs. Ratigns")
plt.show()

Linear regression was applied to analyze the relationship between number of pages and average rating. The slope value of 0.0002187 indicates a very small positive relationship, meaning that increasing the number of pages has almost no impact on the rating. The R² score of 0.0226 shows that only 2.26% of the variation in ratings is explained by the number of pages. This indicates a very weak relationship between book length and rating. Therefore, number of pages is not a strong predictor of average rating.

2. To find relationship between popularity of a book and its rating

In [ ]:
x=np.log1p(df[["ratings_count"]])
y=df["average_rating"]
m=LinearRegression()
m.fit(x,y)
predictions=m.predict(x)
print("Slope:", m.coef_[0])
print("Intercept:", m.intercept_)
print("R^2 score:", m.score(x,y))


- Popularity has a very weak positive effect on rating.
- 97.95% of rating variation depends on other factors

In [ ]:
plt.scatter(x,y,alpha=0.3)
plt.plot(x,predictions, color='red')
plt.xlabel("Log of Ratings Count (Popularity)")
plt.ylabel("Average Rating")
plt.title("Linear Regression: Popularity vs. Rating")
plt.show()

Linear regression was applied to analyze the relationship between book popularity (ratings count) and average rating. A logarithmic transformation was used to reduce skewness in the popularity data. The slope value of 0.01857 indicates a very weak positive relationship between popularity and rating. The R² score of 0.02049 shows that only about 2.05% of the variation in average rating is explained by popularity. This indicates that popularity is not a strong predictor of book rating. Therefore, highly popular books do not necessarily receive higher ratings, and ratings are likely influenced more by subjective factors such as content quality and reader experience.

3. To find relationship between no. of written reviews of a book and its rating

In [ ]:
x=np.log1p(df[["text_reviews_count"]])
y=df["average_rating"]
m=LinearRegression()
m.fit(x,y)
predictions=m.predict(x)
print("Slope:", m.coef_[0])
print("Intercept:", m.intercept_)
print("R^2 score:", m.score(x,y))

- Number of reviews has almost no effect on average rating.
- 99.36% of rating variation is caused by other factors

In [ ]:
plt.scatter(x,y,alpha=0.3)
plt.plot(x,predictions,color='red')
plt.xlabel("Log of Text Reviews Count")
plt.ylabel("Average Rating")
plt.title("Linear Regression: Text Reviews vs. Rating")
plt.show()

Linear regression was applied to analyze the relationship between number of text reviews and average rating. A logarithmic transformation was used to handle skewness in the review count data. The slope value of 0.01270 indicates a very weak positive relationship between review count and rating. The R² score of 0.00641 shows that only 0.64% of the variation in average rating is explained by the number of reviews. This indicates that review count is not a strong predictor of book rating. Therefore, reader engagement in terms of number of reviews does not significantly influence the average rating.

4. To find relationship between no. of reviews and popularity

In [ ]:
x=np.log1p(df[["text_reviews_count"]])
y=np.log1p(df["ratings_count"])
m=LinearRegression()
m.fit(x,y)
predictions=m.predict(x)
print("Slope:", m.coef_[0])
print("Intercept:", m.intercept_)
print("R^2 score:", m.score(x,y))

- The slope value of 1.1706 indicates a strong positive relationship between text review count and ratings count (on log scale).
- The intercept value of 2.0034 represents the predicted log of ratings count when log(text_reviews_count) is zero.
- The R² score tells us that 91.7% of the variation in ratings count is explained by text review count.

In [ ]:
plt.scatter(x,y,alpha=0.3)
plt.plot(x,predictions,color="red")
plt.xlabel("No. of Text Reviews")
plt.ylabel("No. of Ratings")
plt.title("Linear Regression: No. of Reviews vs. No. of Ratings")
plt.show

Linear regression analysis reveals a very strong positive relationship between text review count and ratings count. With an R² value of 0.917, review engagement explains over 91% of the variation in book popularity. This suggests that books that generate more written discussions also tend to attract significantly more ratings. Therefore, review engagement is a strong indicator of popularity.

### 5. To examine whether the number of pages influences the popularity of a book (measured by ratings count).

In [ ]:
x=df[["num_pages"]]
y=np.log1p(df["ratings_count"])
m=LinearRegression()
m.fit(x,y)
predictions=m.predict(x)
print("Slope:", m.coef_[0])
print("Intercept:", m.intercept_)
print("R^2 score:", m.score(x,y))

- Book length has almost no practical effect on popularity.
- The R² score shows that only 1.62% of the variation in ratings count is explained by number of pages.

In [ ]:
plt.scatter(x,y,alpha=0.3)
plt.plot(x,predictions, color="red")
plt.xlabel("Length of book")
plt.ylabel("No. of ratings")
plt.title("Linear Regression: Length of a book vs. No. of ratings")
plt.show()

Linear regression analysis shows that number of pages has a very weak relationship with book popularity. The extremely low slope and R² value indicate that book length does not significantly influence how popular a book becomes. Popularity is likely driven by other external and qualitative factors rather than structural attributes like length. Although the regression line shows a slight upward trend, the scatter plot reveals that the data points are widely dispersed. This indicates that while there is a positive relationship between number of pages and popularity, the relationship is extremely weak. The low R² value confirms that book length does not meaningfully predict popularity.

### 6. To examine whether the number of pages influences the number of text reviews a book receives.

In [ ]:
x=df[["num_pages"]]
y=np.log1p(df[["text_reviews_count"]])
m=LinearRegression()
m.fit(x,y)
predictions=m.predict(x)
print("Slope: ",m.coef_[0])
print("Intercept: ",m.intercept_)
print("R^2 value: ",m.score(x,y))


- Increasing book length has almost no meaningful impact on the number of reviews.
- The R² score shows that only 1.13% of the variation in review count is explained by book length.

In [ ]:
plt.scatter(x,y,alpha=0.3)
plt.plot(x,predictions,color="red")
plt.xlabel("Book Length")
plt.ylabel("No. of text reviews")
plt.title("Linear Regression: Book Length vs. Reviews")
plt.show()

Linear regression analysis shows that number of pages has a very weak positive relationship with review engagement. The low slope and extremely small R² value indicate that book length does not meaningfully influence the number of text reviews. Review engagement is likely driven by qualitative and popularity-based factors rather than structural characteristics like length.

### 7. To examine whether newer books tend to be more popular compared to older books.

In [ ]:

df["publication_date"] = pd.to_datetime(df["publication_date"], errors="coerce")
df["year"] = df["publication_date"].dt.year
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df["average_rating"] = pd.to_numeric(df["average_rating"], errors="coerce")
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=["year", "average_rating"])
x = df[["year"]]
y = df["average_rating"]
predictions = m.predict(x)

m = LinearRegression()
m.fit(x, y)
print("Slope:", m.coef_[0])
print("Intercept:", m.intercept_)
print("R^2 score:", m.score(x, y))

- Newer books are slightly less popular than older books.
- The R^2 score shows that only 0.10% of the variation in popularity is explained by publication year.

In [ ]:
plt.scatter(x,y,alpha=0.3)
plt.plot(x,predictions,color="red")
plt.xlabel("Publication Year")
plt.ylabel("Average Rating")
plt.title("Linear Regression: Publication Year vs. Rating")
plt.show()

Linear regression analysis indicates a very weak negative relationship between publication year and popularity. The extremely low R² value shows that publication year does not meaningfully explain differences in ratings count. This suggests that both older and newer books can achieve high or low popularity depending on other factors such as content quality, author recognition, and reader engagement.

### 8. To analyze whether newer books tend to receive higher average ratings compared to older books.

In [ ]:
df["publication_date"] = pd.to_datetime(df["publication_date"], errors="coerce")
df["year"] = df["publication_date"].dt.year
df_clean = df.dropna(subset=["year", "ratings_count"])
x = df_clean[["year"]]
y = np.log1p(df_clean["ratings_count"])
m = LinearRegression()
m.fit(x, y)
predictions = m.predict(x)
print("Slope:", m.coef_[0])
print("Intercept:", m.intercept_)
print("R^2 score:", m.score(x, y))

- The newer books show a slight upward trend in ratings.
- The R^2 score shows that only 1.58% of the variation in average rating is explained by publication year.

In [ ]:
plt.scatter(x, y, alpha=0.3)
plt.plot(x, predictions)
plt.xlabel("Publication Year")
plt.ylabel("Log of Ratings Count (Popularity)")
plt.title("Linear Regression: Publication Year vs Popularity")
plt.show()

Linear regression analysis shows a weak positive relationship between publication year and average rating. While newer books appear to receive slightly higher ratings on average, the very low R² value indicates that publication year is not a strong predictor of rating. Ratings are likely influenced more by subjective and qualitative factors such as content quality, genre, and reader preference rather than publication time.